# Predicting Student Health Risk — Clean Kaggle Solution

Bu notebook, Kaggle Playground Series S6E7 yarışması için temiz ve paylaşılabilir bir çözüm iskeletidir.

Problem: Öğrencilerin sağlık durumunu üç sınıftan birine tahmin etmek:

- `at-risk`
- `fit`
- `unhealthy`

Yarışma metriği: **Balanced Accuracy**.

Bu metrik normal accuracy'den farklıdır. Her sınıfın recall değerini ayrı ayrı hesaplar ve ortalamasını alır. Bu yüzden hedef sınıf dengesiz olduğunda majority class'a yaslanmak işe yaramaz.

Bu notebook'un amacı:

1. Veriyi hızlı ama doğru okumak.
2. Yarışma metriğine uygun validation kurmak.
3. Tabular veri için güçlü bir CatBoost tabanı oluşturmak.
4. Class imbalance için probability multiplier tuning yapmak.
5. Final submission üretmek.
6. Yarışmadan ve üst seviye çözümlerden çıkarılan dersleri açıkça belgelemek.

Not: Yarışmadaki en iyi private skorumuza ulaşmak için ayrıca multi-seed LightGBM, XGBoost, target encoding, blending ve score lab denemeleri yapılmıştı. Bu notebook, paylaşılabilir ve öğretici ana çözümü sadeleştirilmiş şekilde gösterir.


## 1. Kütüphaneler ve genel ayarlar

İlk hedefimiz kodu hem Kaggle'da hem lokal bilgisayarda çalışabilecek şekilde yazmak.

Pratik kural:

- Kaggle'da veri yolu genelde `/kaggle/input/...`
- Lokalde veri yolu kişisel klasör olur.
- Notebook paylaşılacaksa iki ortamı da desteklemek gerekir.


In [ ]:
from pathlib import Path
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    recall_score,
)

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
N_SPLITS = 3   # Hızlı paylaşım için 3. Final deneylerde 5 veya daha fazla fold denenebilir.
TARGET = "health_condition"
ID_COLUMN = "id"

KAGGLE_DATA_DIR = Path("/kaggle/input/playground-series-s6e7")

LOCAL_DATA_DIR = Path("data")

DATA_DIR = KAGGLE_DATA_DIR if KAGGLE_DATA_DIR.exists() else LOCAL_DATA_DIR
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle").exists() else Path(".")

print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


## 2. Veriyi okuma

Burada üç dosya bekliyoruz:

- `train.csv`: Feature'lar + hedef değişken
- `test.csv`: Sadece feature'lar
- `sample_submission.csv`: Kaggle'ın istediği submission formatı

İlk kontrol:

- Train ve test satır/sütun sayısı
- Hedef değişken train'de var mı?
- Hedef test'te yok mu?
- Submission satır sayısı test ile aynı mı?


In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", sample_submission.shape)

display(train.head())
display(test.head())
display(sample_submission.head())

## 3. Genel veri bakışı

Model kurmadan önce veriye daha genel bakıyoruz.

Bu bölümde amaç:

- Sütun tiplerini görmek
- Sayısal/kategorik yapı hakkında fikir almak
- İlk `describe` tablolarıyla değer aralıklarını anlamak
- Hangi sütunların object/string olduğunu fark etmek

Pratik düşünce:

> `info()` bize veri tiplerini ve eksik olmayan değer sayısını gösterir.  
> `describe()` bize sayısal değerlerin ölçeğini, uç değerlerini ve dağılım özetini verir.

Bu iki komut basit görünür ama modelleme öncesi en hızlı gerçeklik kontrolüdür.


In [ ]:
print("Train info:")
train.info()

print("\nTest info:")
test.info()


In [ ]:
print("Train genel describe:")
display(train.describe(include="all").T)

print("Test genel describe:")
display(test.describe(include="all").T)


## 4. Temel veri kalite kontrolü

Modelden önce dört temel noktaya bakıyoruz:

- Train ve test sütunları uyumlu mu?
- Tamamen tekrar eden satır var mı?
- ID tekrar ediyor mu?
- Eksik değerlerin sayısı ve oranı nedir?

Burada sorun bulursak hemen silmek yerine önce satırları inceleriz.


In [ ]:
print("Train duplicate row:", train.duplicated().sum())
print("Test duplicate row:", test.duplicated().sum())

print("Train duplicate ID:", train[ID_COLUMN].duplicated().sum())
print("Test duplicate ID:", test[ID_COLUMN].duplicated().sum())

print("Train only columns:", sorted(set(train.columns) - set(test.columns)))
print("Test only columns:", sorted(set(test.columns) - set(train.columns)))

duplicate_train_rows = train.loc[train.duplicated(keep=False)]
duplicate_train_ids = train.loc[train[ID_COLUMN].duplicated(keep=False)].sort_values(ID_COLUMN)

if len(duplicate_train_rows) > 0:
    display(duplicate_train_rows.head(10))

if len(duplicate_train_ids) > 0:
    display(duplicate_train_ids.head(10))

In [ ]:
def missing_summary(df):
    summary = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_ratio_pct": df.isna().mean().mul(100).round(2),
        "dtype": df.dtypes.astype(str),
    })
    return summary.sort_values("missing_ratio_pct", ascending=False)

train_missing = missing_summary(train)
test_missing = missing_summary(test)

print("Train missing values:")
display(train_missing.query("missing_count > 0"))

print("Test missing values:")
display(test_missing.query("missing_count > 0"))

## 5. Hedef değişken dağılımı

Bu yarışmanın en kritik noktası hedef dağılımıydı.

`at-risk` sınıfı çok baskın, `fit` ve `unhealthy` azınlık sınıflardı. Bu nedenle:

- Normal accuracy yanıltıcı olur.
- StratifiedKFold gerekir.
- Balanced accuracy takip edilir.
- Class weight veya probability multiplier denenir.


In [ ]:
target_summary = (
    train[TARGET]
    .value_counts()
    .to_frame("count")
    .assign(ratio_percent=lambda x: x["count"] / len(train) * 100)
)

display(target_summary)

plt.figure(figsize=(7, 4))
sns.countplot(data=train, x=TARGET, order=target_summary.index)
plt.title("Target Distribution")
plt.xlabel("Health condition")
plt.ylabel("Count")
plt.show()

majority_class = train[TARGET].mode()[0]
majority_pred = np.full(len(train), majority_class)

print("Majority class:", majority_class)
print("Majority baseline balanced accuracy:", round(balanced_accuracy_score(train[TARGET], majority_pred), 4))


## 6. Sayısal ve kategorik değişkenleri ayırma

Tabular problemlerinde model seçimi ve preprocessing için feature tiplerini ayırırız.

Bu veri setinde:

- Sayısal değişkenler: uyku süresi, kalp ritmi, BMI, kalori, adım sayısı vb.
- Kategorik değişkenler: diyet tipi, stres seviyesi, uyku kalitesi, aktivite seviyesi vb.

CatBoost kategorikleri doğrudan işleyebildiği için bu yarışmada iyi bir başlangıç modelidir.


In [ ]:
feature_columns = [c for c in test.columns if c != ID_COLUMN]
numeric_columns = train[feature_columns].select_dtypes(include=np.number).columns.tolist()
categorical_columns = [c for c in feature_columns if c not in numeric_columns]

print("Feature count:", len(feature_columns))
print("Numeric columns:", numeric_columns)
print("Categorical columns:", categorical_columns)


### 6.1 Aykırı değer kontrolü

IQR yöntemiyle uç değer adaylarının oranını görüyoruz. Bu rapor veri silmez. Çünkü uç bir değer ölçüm hatası olabileceği gibi gerçek ve önemli bir gözlem de olabilir.


In [ ]:
def iqr_outlier_summary(df, columns):
    rows = []

    for column in columns:
        values = df[column].dropna()
        q1 = values.quantile(0.25)
        q3 = values.quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        outlier_mask = (values < lower) | (values > upper)

        rows.append({
            "column": column,
            "lower_bound": lower,
            "upper_bound": upper,
            "outlier_count": outlier_mask.sum(),
            "outlier_ratio_pct": outlier_mask.mean() * 100,
        })

    return pd.DataFrame(rows).sort_values("outlier_ratio_pct", ascending=False)


display(iqr_outlier_summary(train, numeric_columns).round(3))

## 7. Genel görsel EDA

Burada birkaç hızlı grafikle veri hakkında genel his oluşturuyoruz.

Bakmak istediğimiz şeyler:

- Eksik değerler hangi sütunlarda yoğun?
- Sayısal değişkenler yaklaşık nasıl dağılıyor?
- Sayısal değişkenlerde bariz uç değer var mı?
- Kategorik değişkenlerin frekansları dengeli mi?

Bu grafikler karar verdirmek için var. Eğer grafik modelleme kararını değiştirmiyorsa uzatmıyoruz.


In [ ]:
missing_for_plot = train_missing.query("missing_count > 0").sort_values("missing_ratio_pct")

if len(missing_for_plot) > 0:
    plt.figure(figsize=(8, 5))
    plt.barh(missing_for_plot.index, missing_for_plot["missing_ratio_pct"])
    plt.title("Train Missing Ratio by Column")
    plt.xlabel("Missing ratio (%)")
    plt.ylabel("Column")
    plt.tight_layout()
    plt.show()

In [ ]:
for column in numeric_columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 3))

    sns.histplot(train[column], kde=True, ax=axes[0])
    axes[0].set_title(f"{column} distribution")

    sns.boxplot(x=train[column], ax=axes[1])
    axes[1].set_title(f"{column} boxplot")

    plt.tight_layout()
    plt.show()


In [ ]:
for column in categorical_columns:
    plt.figure(figsize=(7, 3))
    order = train[column].fillna("Missing").value_counts().index
    sns.countplot(data=train.fillna("Missing"), x=column, order=order)
    plt.title(f"{column} frequency")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()


## 8. Train-test dağılım kontrolü

Train ve test çok farklıysa cross-validation skoru leaderboard'u iyi temsil etmeyebilir. İlk kontrolde sayısal değişkenlerin ortalama, medyan ve eksik oranlarını karşılaştırıyoruz. Kategorik değişkenlerde ise yalnızca testte görülen yeni sınıfları arıyoruz.


In [ ]:
drift_rows = []

for column in numeric_columns:
    train_std = train[column].std()
    mean_difference = test[column].mean() - train[column].mean()

    drift_rows.append({
        "column": column,
        "train_mean": train[column].mean(),
        "test_mean": test[column].mean(),
        "train_median": train[column].median(),
        "test_median": test[column].median(),
        "normalized_mean_diff": mean_difference / train_std if train_std > 0 else np.nan,
        "missing_ratio_diff": test[column].isna().mean() - train[column].isna().mean(),
    })

numeric_drift = pd.DataFrame(drift_rows)
numeric_drift["abs_normalized_diff"] = numeric_drift["normalized_mean_diff"].abs()
display(numeric_drift.sort_values("abs_normalized_diff", ascending=False).round(4))

In [ ]:
category_rows = []

for column in categorical_columns:
    train_values = set(train[column].dropna().unique())
    test_values = set(test[column].dropna().unique())

    category_rows.append({
        "column": column,
        "train_unique": len(train_values),
        "test_unique": len(test_values),
        "only_in_train": sorted(train_values - test_values),
        "only_in_test": sorted(test_values - train_values),
    })

display(pd.DataFrame(category_rows))

In [ ]:
quantiles = [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]

print("Train numeric summary:")
display(train[numeric_columns].describe(percentiles=quantiles).T)

print("Test numeric summary:")
display(test[numeric_columns].describe(percentiles=quantiles).T)

## 9. Kategorik değişkenlerin hedefle ilişkisi

Burada şunu soruyoruz:

> Bir kategorinin içinde sınıf oranları değişiyor mu?

Örneğin `stress_level = high` olanlarda `unhealthy` oranı artıyorsa bu değişken model için güçlü sinyal taşır.

Bu tabloyu ezberlemek için değil, hangi feature'ların güçlü olduğunu sezmek için kullanıyoruz.


## 10. Feature engineering

Feature engineering'de amaç rastgele yeni sütun üretmek değildir. Önce problem mantığına bakarız:

- Uyku + stres beraber anlamlı olabilir.
- Uyku + BMI beraber sağlık riskini daha iyi açıklayabilir.
- Egzersiz ve adım sayısı birlikte fiziksel aktivite sinyali verir.
- Kategorik kombinasyonlar bazı kuralları yakalayabilir.

Bu yüzden az sayıda, yorumlanabilir interaction üretiyoruz.


In [ ]:
def make_features(df):
    df = df.copy()

    for col in categorical_columns:
        df[col] = df[col].fillna("Missing").astype(str)

    eps = 1e-6

    df["sleep_bmi_interaction"] = df["sleep_duration"] * df["bmi"]
    df["sleep_step_interaction"] = df["sleep_duration"] * df["step_count"]
    df["exercise_step_interaction"] = df["exercise_duration"] * df["step_count"]

    df["sleep_per_bmi"] = df["sleep_duration"] / (df["bmi"] + eps)
    df["calorie_per_exercise"] = df["calorie_expenditure"] / (df["exercise_duration"] + eps)
    df["steps_per_exercise"] = df["step_count"] / (df["exercise_duration"] + eps)

    df["stress_sleep_quality"] = df["stress_level"] + "_" + df["sleep_quality"]
    df["stress_activity"] = df["stress_level"] + "_" + df["physical_activity_level"]
    df["sleep_quality_activity"] = df["sleep_quality"] + "_" + df["physical_activity_level"]

    return df

train_fe = make_features(train)
test_fe = make_features(test)

generated_categorical = ["stress_sleep_quality", "stress_activity", "sleep_quality_activity"]
generated_numeric = [
    "sleep_bmi_interaction",
    "sleep_step_interaction",
    "exercise_step_interaction",
    "sleep_per_bmi",
    "calorie_per_exercise",
    "steps_per_exercise",
]

model_features = feature_columns + generated_numeric + generated_categorical
cat_features = categorical_columns + generated_categorical

print("Model feature count:", len(model_features))
print("Categorical feature count:", len(cat_features))

display(train_fe[model_features].head())


## 11. Validation stratejisi

Bu problemde hedef dengesiz olduğu için random split kullanmak yerine `StratifiedKFold` kullanıyoruz.

Neden?

Çünkü her fold'da sınıf oranları korunmalı. Eğer bir fold'da `fit` veya `unhealthy` oranı saparsa validasyon skoru güvenilmez olur.

Yarışmalarda temel refleks:

1. Yarışma metriğini öğren.
2. Validation'ı o metriğe ve veri yapısına göre kur.
3. Model kararlarını public leaderboard'a göre değil, OOF skoruna göre ver.


In [ ]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(train_fe[TARGET])
class_names = label_encoder.classes_
class_to_index = {name: idx for idx, name in enumerate(class_names)}

print("Class mapping:")
for name, idx in class_to_index.items():
    print(f"{name}: {idx}")

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

for fold, (_, valid_idx) in enumerate(skf.split(train_fe, y), start=1):
    ratio = pd.Series(y[valid_idx]).value_counts(normalize=True).sort_index()
    print(f"Fold {fold} ratio:", ratio.round(4).to_dict())


## 12. CatBoost modeli

CatBoost seçme nedenimiz:

- Kategorik değişkenleri doğal işler.
- Eksik sayısal değerleri yönetebilir.
- Tabular yarışmalarda güçlü baseline verir.
- `auto_class_weights="SqrtBalanced"` ile dengesiz sınıflara daha fazla önem verebilir.

Burada OOF probability ve test probability saklıyoruz. Bu kritik:

- OOF probability ile threshold/multiplier tuning yapılır.
- Test probability fold ortalamasıyla daha stabil tahmin verir.


In [ ]:
from catboost import CatBoostClassifier

X = train_fe[model_features]
X_test = test_fe[model_features]

oof_proba = np.zeros((len(X), len(class_names)))
test_proba = np.zeros((len(X_test), len(class_names)))
fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), start=1):
    print(f"\n===== Fold {fold} / {N_SPLITS} =====")

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]
    y_train = y[train_idx]
    y_valid = y[valid_idx]

    model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.08,
        depth=7,
        loss_function="MultiClass",
        eval_metric="MultiClass",
        auto_class_weights="SqrtBalanced",
        random_seed=RANDOM_STATE + fold,
        allow_writing_files=False,
        verbose=100,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
        eval_set=(X_valid, y_valid),
        early_stopping_rounds=75,
        use_best_model=True,
    )

    valid_proba = model.predict_proba(X_valid)
    oof_proba[valid_idx] = valid_proba
    test_proba += model.predict_proba(X_test) / N_SPLITS

    fold_pred = valid_proba.argmax(axis=1)
    fold_score = balanced_accuracy_score(y_valid, fold_pred)
    fold_scores.append(fold_score)
    print(f"Fold {fold} balanced accuracy: {fold_score:.5f}")

    del model, X_train, X_valid, y_train, y_valid
    gc.collect()

raw_oof_pred = oof_proba.argmax(axis=1)
raw_oof_score = balanced_accuracy_score(y, raw_oof_pred)

print("Fold scores:", np.round(fold_scores, 5))
print("Mean fold score:", round(np.mean(fold_scores), 5))
print("OOF balanced accuracy:", round(raw_oof_score, 5))

## 13. Probability multiplier tuning

Model probability üretir; fakat yarışma metriği balanced accuracy ise raw argmax her zaman en iyi karar olmayabilir.

Bu yarışmada majority class (`at-risk`) çok baskın olduğu için model doğal olarak ona fazla yaslanabilir.

Çözüm:

- Probability'leri sınıf bazında çarparız.
- Azınlık sınıfların olasılığını artırırız.
- En iyi çarpanları sadece OOF üzerinde seçeriz.
- Testte aynı çarpanları uygularız.

Bu fikir, üst seviye çözümlerde de kullanıldı. 2. sıra çözümü bunu Nelder-Mead ile optimize etti; biz burada okunabilir ve güvenli bir grid yaklaşımı gösteriyoruz.


In [ ]:
def apply_class_multipliers(probabilities, multipliers):
    adjusted = probabilities * np.asarray(multipliers)
    return adjusted / adjusted.sum(axis=1, keepdims=True)

fit_idx = class_to_index.get("fit")
unhealthy_idx = class_to_index.get("unhealthy")

search_results = []

fit_grid = np.arange(2.5, 4.6, 0.1)
unhealthy_grid = np.arange(2.5, 4.6, 0.1)

for fit_multiplier in fit_grid:
    for unhealthy_multiplier in unhealthy_grid:
        multipliers = np.ones(len(class_names))
        multipliers[fit_idx] = fit_multiplier
        multipliers[unhealthy_idx] = unhealthy_multiplier

        adjusted_oof = apply_class_multipliers(oof_proba, multipliers)
        adjusted_pred = adjusted_oof.argmax(axis=1)
        score = balanced_accuracy_score(y, adjusted_pred)

        recalls = recall_score(y, adjusted_pred, average=None, labels=np.arange(len(class_names)))

        search_results.append({
            "fit_multiplier": fit_multiplier,
            "unhealthy_multiplier": unhealthy_multiplier,
            "balanced_accuracy": score,
            **{f"{class_names[i]}_recall": recalls[i] for i in range(len(class_names))}
        })

multiplier_results = pd.DataFrame(search_results).sort_values("balanced_accuracy", ascending=False)
display(multiplier_results.head(15))

best_row = multiplier_results.iloc[0]
best_multipliers = np.ones(len(class_names))
best_multipliers[fit_idx] = best_row["fit_multiplier"]
best_multipliers[unhealthy_idx] = best_row["unhealthy_multiplier"]

print("Best multipliers:", dict(zip(class_names, best_multipliers.round(3))))
print("Best OOF balanced accuracy:", round(best_row["balanced_accuracy"], 5))


In [ ]:
adjusted_oof = apply_class_multipliers(oof_proba, best_multipliers)
adjusted_oof_pred = adjusted_oof.argmax(axis=1)

print("Classification report:")
print(classification_report(y, adjusted_oof_pred, target_names=class_names, digits=4))

cm = confusion_matrix(y, adjusted_oof_pred)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm_norm,
    annot=True,
    fmt=".3f",
    xticklabels=class_names,
    yticklabels=class_names,
    cmap="Blues",
)
plt.title("Normalized Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


## 14. Submission üretimi

Final submission için:

1. Fold modellerinden gelen test probability ortalamasını alıyoruz.
2. OOF üzerinde seçtiğimiz class multiplier'ları test probability'lerine uyguluyoruz.
3. En yüksek adjusted probability hangi sınıftaysa onu tahmin ediyoruz.
4. Kaggle formatında CSV kaydediyoruz.


In [ ]:
adjusted_test_proba = apply_class_multipliers(test_proba, best_multipliers)
test_pred_encoded = adjusted_test_proba.argmax(axis=1)
test_pred_labels = label_encoder.inverse_transform(test_pred_encoded)

submission = sample_submission.copy()
submission[TARGET] = test_pred_labels

assert len(submission) == len(test)
assert submission[TARGET].isna().sum() == 0

submission_path = OUTPUT_DIR / "submission_catboost_clean_solution.csv"
submission.to_csv(submission_path, index=False)

print("Saved:", submission_path)
print("Submission shape:", submission.shape)
print("Prediction distribution (%):")
display(submission[TARGET].value_counts(normalize=True).mul(100).round(2))
display(submission.head())

## 15. Bu yarışmada bizim daha ileri score lab'de yaptıklarımız

Bu temiz notebook ana fikri gösteriyor. Yarışmada ayrıca şu deneyleri yaptık:

- CatBoost depth varyantları
- LightGBM
- XGBoost
- Target encoding LightGBM
- Multi-seed training
- Probability blending
- Prior correction
- Class-specific multiplier gridleri
- Basit stacking denemeleri
- Disagreement / arbiter denemeleri
- Missing indicator denemeleri

Son private leaderboard sonucumuz:

- Private score: yaklaşık `0.95043`
- Rank: `173`
- Winning score: yaklaşık `0.95085`

Bu yarışmada skorlar aşırı sıkışıktı. İlk sıralarla fark çok küçük olduğu için son aşamada model çeşitliliği, class-specific blending ve daha stabil OOF optimizasyonu belirleyici oldu.


## 16. 2. sıra çözümünden öğrendiklerimiz

2. sıra çözümünün ana fikri şuydu:

### 13.1 Daha büyük model havuzu

18 base predictor kullanıldı:

- LightGBM
- XGBoost
- CatBoost
- FT-Transformer
- RealMLP
- HGBC

Bizim model havuzumuz daha küçüktü. En büyük farklardan biri buydu.

### 13.2 Class-specific ensemble weights

Klasik ensemble'da her modele tek ağırlık verilir:

```text
final_probability = 0.5 * model_A + 0.5 * model_B
```

Ama 2. sıra çözümü her modelin her sınıf için ayrı ağırlığını optimize etti:

```text
18 model x 3 sınıf = 54 ağırlık
```

Bu daha güçlüdür çünkü bir model `fit` sınıfında iyi, başka bir model `unhealthy` sınıfında iyi olabilir.

### 13.3 Önce LogLoss, sonra Balanced Accuracy

Üst seviye çözüm iki aşamalıydı:

1. SLSQP ile OOF LogLoss optimize edildi.
2. Nelder-Mead ile class multiplier optimize edilerek Balanced Accuracy artırıldı.

Bu önemli bir yarışma prensibidir:

> Önce probability kalitesini iyileştir, sonra yarışma metriğine göre karar sınırını ayarla.

### 13.4 Public leaderboard'a fazla güvenmemek

Public leaderboard azınlık sınıflarda gürültülü olabilir. Bu yüzden güvenilir karar:

- OOF score
- fold stability
- prediction distribution
- model çeşitliliği

birlikte değerlendirilerek verilir.
